# 📗 Neo4j 시작하기: 노드·관계·속성 모델

앞서 Movies 그래프를 불러와 첫 조회를 해 봤습니다. 이번 시간에는 이 그래프가 **무엇으로 이루어져 있는지** 를 뜯어봅니다. 노드에 붙는 **레이블**, 관계의 **방향**, 그리고 노드뿐 아니라 **관계에도 붙는 속성**까지. 마지막으로 이 그래프 모델을 익숙한 **표(RDB)** 와 견주어 정리합니다.

여전히 쿼리는 모두 제공됩니다. 여러분은 **실행하고 결과를 읽으며 모델을 이해**하면 됩니다.

## ⏪ 지난 시간

- Neo4j 를 설치·연결하고 Movies 예제 그래프를 불러왔습니다.
- `run_cypher(...)` 결과는 **dict 의 리스트**라는 걸 확인했습니다. 오늘도 그 결과를 파이썬으로 읽습니다.

**오늘의 목표**

- [ ] **노드**와 **레이블**(Person·Movie)이 무엇인지 결과로 확인한다.
- [ ] **관계**에 **방향**이 있음을 이해한다(배우 → 영화), 같은 종류끼리도 이어짐을 본다(`FOLLOWS`).
- [ ] 한 쌍을 잇는 **관계가 여러 개**일 수 있음을 확인한다(감독이면서 출연).
- [ ] **속성**이 노드뿐 아니라 **관계에도** 붙는다는 걸 확인한다(`roles`·`rating`).
- [ ] 그래프 모델을 **RDB 표**와 대응시키고, 모델링 원칙(개체=노드·동사=관계·수식=속성)을 익힌다.
- [ ] 설계를 **레이블·관계·노드 속성·관계 속성** 네 칸으로 적어 본다.

---
## 먼저: 오늘 나오는 Cypher 를 읽는 법

오늘도 쿼리는 전부 주어집니다. **직접 쓰는 법은 다음 단원**에서 배웁니다. 다만 결과를 읽으려면 그 쿼리가 무엇을 물었는지는 알아야 하니, 오늘 나오는 조각만 먼저 훑어 둡니다.

<img src="images/cypher_basics.png" width="820">

| 조각 | 읽는 법 | 오늘 나오는 예 |
|---|---|---|
| `MATCH` | 그래프에서 **찾을 모양**을 적는다 | `MATCH (m:Movie)` |
| `(변수:레이블)` | **노드** 하나. 콜론 뒤가 레이블, 앞은 뒤에서 다시 쓸 이름 | `(p:Person)`. 다시 쓸 일이 없으면 `(:Movie)`, 종류도 안 가리면 `()` |
| `-[변수:관계]->` | **관계**와 그 **방향**. 대괄호 안이 관계 이름이고, **앞뒤 붙임표와 화살표까지가 한 덩어리** | `-[:ACTED_IN]->`. 속성을 꺼내려면 `-[r:ACTED_IN]->`, 종류를 안 가리면 `-[r]->` |
| 화살촉 방향 | **화살촉이 향하는 쪽**이 방향이다. 오른쪽 노드를 먼저 적으면 화살촉을 왼쪽으로 돌린다 | `(:Movie {title:'...'})<-[:ACTED_IN]-(p:Person)`. 화살촉을 빼면(`-[:ACTED_IN]-`) 방향을 안 가린다 |
| `{속성:값}` | 노드나 관계 표기 **안에서 바로 거는 조건** | `(:Movie {title:'The Matrix'})` |
| `RETURN ... AS 이름` | 돌려받을 값과 **별칭**. 별칭이 결과 dict 의 **키**가 된다 | `RETURN p.name AS actor` |
| `ORDER BY` · `LIMIT` | 정렬과 개수 제한. 뒤에 `DESC` 를 붙이면 내림차순 | `ORDER BY p.name LIMIT 5`, `ORDER BY c DESC` |
| `count(...)` · `type(r)` | 개수 세기 · 관계 이름 꺼내기 | `count(m)`, `type(r)` |
| `CALL ... YIELD` | 서버에 **내장된 프로시저**를 부르고 받을 칸을 고른다 | `CALL db.labels() YIELD label` |

> 화살표 방향이 곧 사실의 방향입니다. `(:Person)-[:ACTED_IN]->(:Movie)` 는 "사람이 영화에 출연했다" 이고, 거꾸로 적으면 성립하지 않습니다. 2절에서 이 점을 결과로 확인합니다.

> 2-2절에 딱 한 번 `WITH`·`collect`·`WHERE` 가 함께 나옵니다. 한 쌍에 걸린 관계 이름을 모아 2종 이상만 남기려고 쓴 것이고, **문법 자체는 뒤 단원에서 정식으로 배웁니다.** 오늘은 결과만 읽으면 됩니다.

아래 준비 셀을 먼저 실행하세요. 그다음 점검 셀로 Movies 가 적재됐는지 확인합니다.

In [1]:
# Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 이 단원은 그래프를 조회만 합니다(그래프를 바꾸지 않습니다).
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase   # 파이썬용 공식 드라이버. 이 클래스로 접속 통로를 연다

# 1) 접속 정보 읽기: .env 에 적힌 값을 환경변수로 올린다(파일이 없으면 조용히 넘어간다)
load_dotenv(".env", override=True)       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 두 번째 인자는 .env 에 그 키가 없을 때 쓰는 기본값이다(로컬 Desktop 의 표준 주소·사용자).
# .env 를 못 읽어도 에러가 아니라 이 값으로 조용히 넘어가니, 이 셀 마지막 줄에 찍히는
# 주소가 실습 전용 DB 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")
# 2) 드라이버 만들기: 접속 통로 하나를 노트북 전체가 나눠 쓴다(쿼리마다 새로 만들지 않는다)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 실제로 붙어 본다. 인스턴스가 꺼져 있거나 비밀번호가 틀리면 여기서 에러가 난다


# 3) 수업 내내 쓰는 헬퍼: 쿼리를 보내고 결과를 파이썬 자료형으로 바꿔 준다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # 세션은 with 블록을 벗어나면 자동으로 닫힌다. record.data() 가 결과 한 행을 dict 로 바꾼다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)   # 이 줄이 찍히면 연결까지 성공한 것이다

Neo4j 연결: bolt://localhost:7687


In [2]:
# Movies 그래프가 적재돼 있는지 점검: 실행만 하세요(그래프를 바꾸지 않습니다).
# MATCH (n) 은 레이블을 가리지 않고 모든 노드를 고른다. count(n) 결과는 한 행이라 [0] 으로 dict 를 꺼낸다
_n = run_cypher("MATCH (n) RETURN count(n) AS cnt")[0]["cnt"]   # 이름 앞 밑줄은 이 셀에서만 쓰는 임시 변수라는 표시
print("연결된 그래프의 노드 수:", _n)   # 171 이면 준비 완료, 0 이면 아직 적재 전이다

연결된 그래프의 노드 수: 171


---
# 1. 노드와 레이블

노드와 **레이블**(종류 이름표)은 지난 단원 LPG 에서 배웠습니다. 오늘 새로운 것은 그것을 **실제 데이터베이스에 물어서** 확인하는 일입니다. Movies 그래프의 레이블은 딱 두 종류(**`Person`**·**`Movie`**)인데, 그 사실을 그림이 아니라 **쿼리 결과로** 확인해 봅니다.

먼저 오늘 들여다볼 그래프의 **전체 모습**입니다. 노드 2종류와 이들을 잇는 관계 6종류(괄호 안은 실제 개수).

<img src="images/movies_schema.png" width="760">

## 확인해 봅시다
제공 쿼리로 이 그래프에 어떤 레이블이 있는지, 그리고 각 레이블의 노드가 몇 개인지 봅니다. 위 그림의 숫자와 같은지 눈으로 맞춰 보세요.

In [3]:
# 이 그래프에 있는 레이블(노드 종류) 목록(실행만 하세요).
# db.labels(): 이 데이터베이스에 등록된 레이블 목록을 알려 주는 내장 프로시저
# 노드를 하나씩 뒤지는 게 아니라 서버가 들고 있는 목록을 그대로 받아 온다
# (그래서 노드가 하나도 없는 레이블이 목록에 남아 있을 수 있다)
labels = run_cypher("CALL db.labels() YIELD label RETURN label ORDER BY label")
print(labels)   # 이 그래프의 노드 종류는 두 가지뿐이다

[{'label': 'Movie'}, {'label': 'Person'}]


In [4]:
# 레이블별 노드 개수(실행만 하세요).
# 레이블 이름만 다르고 나머지는 똑같은 쿼리를 두 번 던진다
# 집계 결과는 한 행이라 [0] 으로 dict 를 꺼내고, AS 로 붙인 이름 c 로 숫자를 읽는다
person_count = run_cypher("MATCH (p:Person) RETURN count(p) AS c")[0]['c']
movie_count = run_cypher("MATCH (m:Movie) RETURN count(m) AS c")[0]['c']
print('Person 노드:', person_count)
print('Movie 노드:', movie_count)     # 둘을 더하면 점검 셀에서 본 전체 노드 수가 된다

Person 노드: 133
Movie 노드: 38


> 레이블은 노드의 **종류**입니다. `:Person` 은 사람, `:Movie` 는 영화. 이렇게 종류로 나눠 두면 "영화만", "사람만" 같은 조회가 쉬워집니다.

> 이 목록에 **다른 레이블이 함께 나온다면** 그 인스턴스에 예전에 다른 실습 데이터를 넣었던 것입니다. 그 레이블의 노드는 0개라 아래 개수에는 영향이 없습니다.

---
# 2. 관계와 방향

관계와 **방향**도 지난 단원에서 배웠습니다. 오늘 볼 것은 그 방향이 실제 데이터베이스에 **어느 쪽으로 저장돼 있는지**입니다. `ACTED_IN` 은 **배우 → 영화** 로 향합니다(사람이 영화에 출연한 것이지, 영화가 사람에 출연한 게 아니니까요).

## 확인해 봅시다
이 그래프에 어떤 관계 종류가 있는지, 그리고 `DIRECTED`(감독) 관계가 어느 방향으로 이어지는지 봅니다.

In [5]:
# 이 그래프의 관계 종류 목록(실행만 하세요).
# db.relationshipTypes(): db.labels() 의 관계 버전. 이 데이터베이스에 등록된 관계 이름을 알려 준다
rel_types = run_cypher("CALL db.relationshipTypes() YIELD relationshipType "
                       "RETURN relationshipType AS rel ORDER BY rel")
print(rel_types)   # 관계 종류는 여섯 가지다

[{'rel': 'ACTED_IN'}, {'rel': 'DIRECTED'}, {'rel': 'FOLLOWS'}, {'rel': 'PRODUCED'}, {'rel': 'REVIEWED'}, {'rel': 'WROTE'}]


In [6]:
# 관계 종류별 개수(실행만 하세요). 위 그림의 괄호 안 숫자와 맞춰 보세요.
# ()-[r]->() 는 양쪽 노드의 종류를 가리지 않고 관계만 고른다. type(r) 이 그 관계의 이름이다
rel_counts = run_cypher("MATCH ()-[r]->() RETURN type(r) AS rel, count(r) AS c "
                        "ORDER BY c DESC")
for row in rel_counts:
    print(row)   # 다 더하면 그림에 적힌 관계 수와 같다

{'rel': 'ACTED_IN', 'c': 172}
{'rel': 'DIRECTED', 'c': 44}
{'rel': 'PRODUCED', 'c': 15}
{'rel': 'WROTE', 'c': 10}
{'rel': 'REVIEWED', 'c': 9}
{'rel': 'FOLLOWS', 'c': 3}


In [7]:
# 방향 확인: (사람)-[:DIRECTED]->(영화) 패턴으로 감독-영화 쌍을 가져옵니다(실행만 하세요).
# 패턴이 곧 조건이다: 사람에서 영화로 향하는 DIRECTED 관계만 걸리고, 반대 방향은 걸리지 않는다
# ORDER BY 를 두 칸(감독 이름, 제목)으로 준 이유는 한 감독이 여러 편을 맡았을 때 순서를 고정하기 위해서다
directed = run_cypher("MATCH (p:Person)-[:DIRECTED]->(m:Movie) "
                      "RETURN p.name AS director, m.title AS movie ORDER BY p.name, m.title LIMIT 5")
for row in directed:
    print(row)

{'director': 'Cameron Crowe', 'movie': 'Jerry Maguire'}
{'director': 'Chris Columbus', 'movie': 'Bicentennial Man'}
{'director': 'Clint Eastwood', 'movie': 'Unforgiven'}
{'director': 'Danny DeVito', 'movie': 'Hoffa'}
{'director': 'Frank Darabont', 'movie': 'The Green Mile'}


> 화살표 `-[:DIRECTED]->` 의 방향은 **감독(사람)에서 영화로** 향합니다. 방향이 있어서 "이 사람이 감독한 영화"와 "이 영화를 감독한 사람"을 구분해 물을 수 있습니다. Movies 의 관계 종류는 `ACTED_IN`(출연)·`DIRECTED`(감독)·`PRODUCED`(제작)·`WROTE`(각본)·`REVIEWED`(평가)·`FOLLOWS`(팔로우) 입니다.

### 🖐️ 함께 따라하기: 제작자 이름만 중복 없이 모으기

이번에는 **제작**(`PRODUCED`) 관계입니다. 앞 셀이 감독-영화 쌍을 본 것처럼 **제작자-영화 쌍**을 앞 5행만 가져와 `produced` 에 담고, 거기서 **제작자 이름만** 중복 없이 모은 집합 **`producer_set`** 을 만들어 출력하세요.

**집합(set)** 으로 만드는 이유가 여기서 드러납니다. 한 제작자가 여러 편을 맡으면 이름이 여러 번 나오는데, 집합은 그것을 한 번으로 줄입니다. **행 수와 집합 크기를 함께 찍어** 견줘 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 사람에서 영화로 향하는 PRODUCED 관계를 앞 5행만 가져와 produced 에 담는다
# 2) produced 의 각 row 에서 제작자 이름을 꺼내 집합(set) 으로 모아 producer_set 을 만든다
# 3) 행 수와 집합 크기를 함께 출력해 견준다

In [8]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 사람에서 영화로 향하는 PRODUCED 관계를 앞 5행만 가져와 produced 에 담는다
# 2) produced 의 각 row 에서 제작자 이름을 꺼내 집합(set) 으로 모아 producer_set 을 만든다
# 3) 행 수와 집합 크기를 함께 출력해 견준다


directed = run_cypher("MATCH (p:Person)-[:PRODUCED]->(m:Movie) "
                      "RETURN p.name AS producer, m.title AS movie ORDER BY p.name, m.title LIMIT 5")

for row in directed:
    print(row)

{'producer': 'Cameron Crowe', 'movie': 'Jerry Maguire'}
{'producer': 'Joel Silver', 'movie': 'Ninja Assassin'}
{'producer': 'Joel Silver', 'movie': 'Speed Racer'}
{'producer': 'Joel Silver', 'movie': 'The Matrix'}
{'producer': 'Joel Silver', 'movie': 'The Matrix Reloaded'}


## 같은 종류끼리 잇는 관계: FOLLOWS
지금까지 본 관계는 모두 **사람 → 영화**였습니다. 그래서 "관계는 서로 다른 종류의 노드를 잇는 것"이라고 오해하기 쉽습니다. 이 그래프에는 **사람 → 사람** 관계도 하나 있습니다. `FOLLOWS`(팔로우)입니다.

In [10]:
# 사람에서 사람으로 향하는 유일한 관계 FOLLOWS(실행만 하세요).
# 패턴 양쪽 끝이 둘 다 :Person 이다. 관계가 서로 다른 레이블만 잇는 게 아니라는 증거다
# LIMIT 을 안 건 이 관계가 몇 개 없어서다. 전부 받아도 짧다
follows = run_cypher("MATCH (a:Person)-[:FOLLOWS]->(b:Person) "
                     "RETURN a.name AS follower, b.name AS followee ORDER BY follower")
for row in follows:
    print(row)   # follower 가 followee 를 팔로우한다. 거꾸로는 성립하지 않는다

{'follower': 'Angela Scope', 'followee': 'Jessica Thompson'}
{'follower': 'James Thompson', 'followee': 'Jessica Thompson'}
{'follower': 'Paul Blythe', 'followee': 'Angela Scope'}


In [11]:
# 사람에서 사람으로 향하는 유일한 관계 FOLLOWS(실행만 하세요).
# 패턴 양쪽 끝이 둘 다 :Person 이다. 관계가 서로 다른 레이블만 잇는 게 아니라는 증거다
# LIMIT 을 안 건 이 관계가 몇 개 없어서다. 전부 받아도 짧다
follows = run_cypher("MATCH (a:Person)-[:FOLLOWS]->(b:Person) "
                     "RETURN a.name AS follower, b.name AS followee ORDER BY follower")
for row in follows:
    print(row)   # follower 가 followee 를 팔로우한다. 거꾸로는 성립하지 않는다

{'follower': 'Angela Scope', 'followee': 'Jessica Thompson'}
{'follower': 'James Thompson', 'followee': 'Jessica Thompson'}
{'follower': 'Paul Blythe', 'followee': 'Angela Scope'}


> 관계는 **같은 레이블끼리도** 이어지고, 그때도 **방향이 뜻을 바꿉니다**. "A 가 B 를 팔로우"와 "B 가 A 를 팔로우"는 완전히 다른 사실이죠. 앞의 그림에서 Person 에 붙어 자기에게 돌아오던 화살표가 바로 이 관계입니다.

## 화살촉이 향하는 쪽이 방향입니다

`FOLLOWS` 는 양 끝이 둘 다 `Person` 이라 **뜻을 가르는 것이 오직 화살촉 방향**입니다. **Angela Scope** 한 사람을 두고 화살촉만 바꿔 가며 네 번 물어봅시다.

적는 순서를 바꾸는 것과 화살촉을 돌리는 것은 **전혀 다른 일**입니다. 그 차이가 결과로 드러납니다.

In [12]:
# 같은 자리에서 화살촉만 바꿔 봅니다(실행만 하세요).
# 1) Angela Scope 를 팔로우하는 사람. 화살촉이 Angela Scope 를 향한다
followers = run_cypher("MATCH (a:Person)-[:FOLLOWS]->(:Person {name:'Angela Scope'}) "
                       "RETURN a.name AS name ORDER BY a.name")
# 2) 같은 질문을 Angela Scope 부터 적었다. 화살촉을 왼쪽으로 돌렸을 뿐 가리키는 사실은 1)과 같다
followers_2 = run_cypher("MATCH (:Person {name:'Angela Scope'})<-[:FOLLOWS]-(a:Person) "
                         "RETURN a.name AS name ORDER BY a.name")
# 3) 화살촉을 빼면 방향을 가리지 않아 양쪽이 다 걸린다
both_ways = run_cypher("MATCH (:Person {name:'Angela Scope'})-[:FOLLOWS]-(a:Person) "
                       "RETURN a.name AS name ORDER BY a.name")
# 4) 화살촉을 반대로 돌리면 'Angela Scope 가 팔로우하는 사람' 이 되어 답이 아예 달라진다
followees = run_cypher("MATCH (:Person {name:'Angela Scope'})-[:FOLLOWS]->(a:Person) "
                       "RETURN a.name AS name ORDER BY a.name")
for label, rows in [('1) Angela Scope 를 팔로우하는 사람', followers),
                    ('2) 같은 질문, 적는 순서만 바꿈', followers_2),
                    ('3) 화살촉 없음(양쪽 다)', both_ways),
                    ('4) 화살촉을 반대로', followees)]:
    print(label, ':', [row['name'] for row in rows])

1) Angela Scope 를 팔로우하는 사람 : ['Paul Blythe']
2) 같은 질문, 적는 순서만 바꿈 : ['Paul Blythe']
3) 화살촉 없음(양쪽 다) : ['Jessica Thompson', 'Paul Blythe']
4) 화살촉을 반대로 : ['Jessica Thompson']


### 🖐️ 함께 따라하기: 영화 한 편의 출연진 세기

Cypher 를 쓰는 법은 다음 단원에서 배우지만, **앞 셀의 쿼리를 고쳐 다시 묻는 것**은 지금도 할 수 있습니다.

**영화 `The Matrix` 에 출연한 배우**를 모두 가져와 `matrix_cast` 에 담고, 몇 명인지 출력하세요.

- 2절 첫 제공 셀의 `DIRECTED` 쿼리를 바탕으로, 관계를 `:ACTED_IN` 으로 바꿉니다.
- 도착 노드에 **조건을 직접 적어** 그 영화만 남깁니다. 패턴 안 중괄호가 그 자리입니다: `(:Movie {title:'The Matrix'})`. 뒤에서 다시 쓸 일이 없으니 변수 이름은 붙이지 않아도 됩니다.
- 사람 이름만 돌려받고 `ORDER BY`·`LIMIT` 은 빼세요. **결과 한 행이 배우 한 명**이라 행 수가 곧 인원수입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 사람에서 영화로 향하는 ACTED_IN 패턴을 쓰되, 영화 노드에 title 조건을 직접 적는다
# 2) 사람 이름만 돌려받아 matrix_cast 에 담는다
# 3) len(matrix_cast) 로 인원수를 출력한다

In [13]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 사람에서 영화로 향하는 ACTED_IN 패턴을 쓰되, 영화 노드에 title 조건을 직접 적는다
# 2) 사람 이름만 돌려받아 matrix_cast 에 담는다
# 3) len(matrix_cast) 로 인원수를 출력한다

matrix_cast = run_cypher("MATCH (p:Person)-[:ACTED_IN]->(:Movie {title:'The Matrix'}) "
                       "RETURN p.name AS name ORDER BY p.name")

print(matrix_cast)
print(len(matrix_cast))

[{'name': 'Carrie-Anne Moss'}, {'name': 'Emil Eifrem'}, {'name': 'Hugo Weaving'}, {'name': 'Keanu Reeves'}, {'name': 'Laurence Fishburne'}]
5


---
# 2-2. 같은 두 노드를 잇는 관계가 여러 개일 때

한 사람이 어떤 영화를 **감독하면서 출연**하기도 하고, **각본까지 쓰고 제작도** 합니다. 이럴 때 그래프는 두 노드 사이에 **화살표를 하나 더 그을 뿐**입니다. 표라면 출연 표와 감독 표를 따로 두고 **두 번 조인**해야 알 수 있는 사실이죠.

그리고 각 관계는 **자기 속성을 따로** 가집니다. 출연에는 배역(`roles`)이 붙지만 감독에는 붙지 않습니다. 한 줄로 뭉쳐 놓으면 이런 구분이 사라집니다.

<img src="images/multi_relationship.png" width="760">

## 확인해 봅시다
한 (사람, 영화) 쌍을 잇는 관계가 **2종 이상**인 경우를 찾아봅니다.

In [14]:
# 한 쌍을 잇는 관계가 2종 이상인 (사람, 영화) 쌍(실행만 하세요).
# [r] 처럼 관계 이름을 비워 두면 종류를 가리지 않고 전부 걸린다. type(r) 이 그 관계의 이름이다
# WITH: 사람·영화 쌍으로 묶어 관계 이름을 collect 로 리스트에 모으는 중간 단계
#       (WITH 와 collect 문법 자체는 뒤 단원에서 정식으로 배운다. 지금은 결과만 읽으면 된다)
# collect 에 DISTINCT 를 붙여 같은 종류를 두 번 담지 않게 했다. 그래서 size(types) 가 곧 관계 '종류' 수다
multi = run_cypher("MATCH (p:Person)-[r]->(m:Movie) WITH p, m, collect(DISTINCT type(r)) AS types "
                   "WHERE size(types) >= 2 RETURN p.name AS person, m.title AS movie, types "
                   "ORDER BY size(types) DESC, p.name")
print('관계가 2종 이상인 (사람, 영화) 쌍:', len(multi))

관계가 2종 이상인 (사람, 영화) 쌍: 12


In [15]:
for row in multi[:5]:   # 12쌍을 다 찍지 않고 앞 5쌍만 본다. 정렬을 DESC 로 해서 관계가 3종인 쌍이 먼저 온다
    print(row)   # 앞 2줄이 types 에 3종(감독·제작·각본)이 담긴 쌍이다

{'person': 'Cameron Crowe', 'movie': 'Jerry Maguire', 'types': ['DIRECTED', 'PRODUCED', 'WROTE']}
{'person': 'Nancy Meyers', 'movie': "Something's Gotta Give", 'types': ['DIRECTED', 'PRODUCED', 'WROTE']}
{'person': 'Aaron Sorkin', 'movie': 'A Few Good Men', 'types': ['ACTED_IN', 'WROTE']}
{'person': 'Clint Eastwood', 'movie': 'Unforgiven', 'types': ['ACTED_IN', 'DIRECTED']}
{'person': 'Danny DeVito', 'movie': 'Hoffa', 'types': ['ACTED_IN', 'DIRECTED']}


> 실행하면 이런 쌍이 **12쌍**, 그중 관계가 **3종**인 쌍이 **2쌍** 나옵니다. 예를 들어 **Clint Eastwood 는 Unforgiven 을 감독하면서 출연**했습니다. 두 사실이 화살표 두 개로 나란히 저장됩니다. Cameron Crowe 는 Jerry Maguire 를 각본·제작·감독까지 해서 화살표가 세 개입니다.

---
# 3. 속성: 노드에도, 관계에도

노드와 관계에 딸린 값을 **속성**이라고 합니다. 사람 노드에는 `name`·`born`(출생연도), 영화 노드에는 `title`·`released`(개봉연도)·`tagline` 같은 속성이 있죠. 여기까지는 표의 '열'과 비슷합니다.

**속성이 관계에도 붙는다**는 것도 지난 단원에서 봤습니다. 오늘은 그 값을 실제로 꺼내 봅니다. 예를 들어 배우가 영화에서 맡은 **배역(`roles`)** 은 사람의 속성도, 영화의 속성도 아닙니다. **그 출연(`ACTED_IN` 관계) 자체의 속성**입니다(같은 배우라도 영화마다 배역이 다르니까요). 평가 점수 `rating` 도 마찬가지로 `REVIEWED` **관계의 속성**입니다.

<img src="images/node_anatomy.png" width="760">

> 값이 **어디에 붙어 있는지**가 모델의 전부입니다. 이 그림을 머릿속에 두고 아래 결과를 읽어 보세요.

## 확인해 봅시다

In [16]:
# 노드 속성: 영화의 title·released, 사람의 name·born(실행만 하세요).
# {title:'The Matrix'} 로 영화 한 편만 집어 속성 세 개를 꺼낸다. 결과가 한 행이라 [0] 으로 dict 를 본다
movie_props = run_cypher("MATCH (m:Movie {title:'The Matrix'}) "
                         "RETURN m.title AS title, m.released AS released, m.tagline AS tagline")
print('영화 노드 속성:', movie_props[0])

영화 노드 속성: {'title': 'The Matrix', 'released': 1999, 'tagline': 'Welcome to the Real World'}


In [17]:
# 같은 그래프라도 레이블이 다르면 속성 이름이 다르다(영화는 title·released, 사람은 name·born)
person_props = run_cypher("MATCH (p:Person {name:'Tom Hanks'}) "
                          "RETURN p.name AS name, p.born AS born")
print('사람 노드 속성:', person_props[0])

사람 노드 속성: {'name': 'Tom Hanks', 'born': 1956}


In [18]:
# 관계 속성 1) ACTED_IN 의 roles(배역 목록)를 꺼냅니다(실행만 하세요).
# 관계에 변수 r 을 붙여야 r.roles 로 관계의 속성을 꺼낼 수 있다(노드가 아니라 연결에 붙은 값이다)
roles_rows = run_cypher("MATCH (p:Person)-[r:ACTED_IN]->(m:Movie {title:'Cloud Atlas'}) "
                        "RETURN p.name AS actor, r.roles AS roles ORDER BY p.name")
for row in roles_rows:
    print(row)   # roles 값이 리스트라 배역이 여러 개 담겨 있다

{'actor': 'Halle Berry', 'roles': ['Luisa Rey', 'Jocasta Ayrs', 'Ovid', 'Meronym']}
{'actor': 'Hugo Weaving', 'roles': ['Bill Smoke', 'Haskell Moore', 'Tadeusz Kesselring', 'Nurse Noakes', 'Boardman Mephi', 'Old Georgie']}
{'actor': 'Jim Broadbent', 'roles': ['Vyvyan Ayrs', 'Captain Molyneux', 'Timothy Cavendish']}
{'actor': 'Tom Hanks', 'roles': ['Zachry', 'Dr. Henry Goose', 'Isaac Sachs', 'Dermot Hoggins']}


> `roles` 가 **리스트**인 점을 보세요. 한 배우가 한 영화에서 **여러 배역**을 맡을 수 있어서입니다(예: Cloud Atlas). 그리고 이 값이 사람이나 영화가 아니라 **둘을 잇는 관계**에 붙어 있다는 게 핵심입니다. "어디에 붙일까"를 정하는 게 바로 모델링입니다.

In [ ]:
# 관계 속성 2) REVIEWED 의 rating(점수)·summary(한 줄 평)을 꺼냅니다(실행만 하세요).
# rating 도 사람이나 영화가 아니라 REVIEWED 관계에 붙은 값이다
# 한 영화에 붙은 리뷰들을 점수 높은 순으로 본다. 관계 속성으로도 정렬할 수 있다
rating_rows = run_cypher("MATCH (p:Person)-[r:REVIEWED]->(:Movie {title:'The Da Vinci Code'}) "
                         "RETURN p.name AS reviewer, r.rating AS rating, r.summary AS summary "
                         "ORDER BY r.rating DESC")
# 한 관계에 속성이 둘이다(rating 과 summary). 관계도 노드처럼 여러 값을 들고 다닌다
for row in rating_rows:
    print(row)   # 같은 영화인데 사람마다 점수도 한 줄 평도 다르다

> `rating` 도 마찬가지입니다. 같은 **The Da Vinci Code** 인데 리뷰어마다 점수가 다릅니다(68 대 65). 점수가 영화 노드에 붙어 있었다면 한 편에 하나뿐이라 "누가 매긴 점수인가"를 담을 수 없습니다. **값이 사람마다 달라지면 그 값은 연결에 붙습니다.**

> **속성은 늘 있는 게 아닙니다.** 이 그래프의 사람 133명 중 일부는 `born`(출생연도)이 없습니다(그래프에 그 정보가 채워지지 않은 것). 표라면 빈 칸(NULL)이 생기지만, 그래프에서는 그냥 **그 속성이 없는 노드**가 됩니다.

### 🖐️ 함께 따라하기: 다른 영화의 배역 수 세기

영화 **The Polar Express** 의 출연 배우와 **배역 리스트**를 가져와 `polar_rows` 에 담으세요. 그다음 각 배우가 맡은 **배역 수**를 `{배우이름: 배역수}` 형태의 dict **`polar_roles`** 로 만들어 출력하세요.

- 배역은 `ACTED_IN` **관계의 속성** 이라, 관계에 변수를 붙여야 꺼낼 수 있습니다(앞 셀과 같은 모양).
- 배역 수는 그 리스트의 길이입니다.

> 이 영화는 출연 배우가 **한 명**이라 dict 도 한 칸입니다. 그래도 키를 **배우 이름**으로 잡아 두면, 같은 코드를 Cloud Atlas 처럼 배우가 여럿인 영화에 그대로 돌렸을 때 배우마다 한 칸씩 생깁니다. 한 배우가 배역을 몇 개까지 맡을 수 있는지도 함께 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) The Polar Express 에 출연한 배우 이름과 배역 리스트를 가져와 polar_rows 에 담는다
# 2) 각 row 에서 배우 이름을 키로, 배역 리스트의 길이를 값으로
# 3) polar_roles dict 를 만들어 출력한다

---
# 4. 표(RDB)와의 대응, 그리고 모델링 원칙

## 그래프 ↔ RDB 대응표
RDB 와 그래프를 견주는 일은 지난 단원에서도 했습니다. 오늘 새로운 것은 **방금 조회한 실제 값**을 표의 어느 자리에 놓을지 한 칸씩 짝지어 보는 것입니다.

| 그래프(Neo4j) | 관계형 DB(표) | Movies 예 |
|---|---|---|
| 노드(Node) | 행(Row) | 영화 한 편, 사람 한 명 |
| 레이블(Label) | 테이블 이름 | `Person`, `Movie` |
| 노드 속성 | 열(Column) 값 | `title`, `released`, `born` |
| 관계(Relationship) | 외래키 / 조인 테이블 | `ACTED_IN`, `DIRECTED` |
| 관계 속성 | 조인 테이블의 추가 열 | `roles`, `rating` |

<img src="images/rdb_vs_graph.png" width="760">

> 표에서 **조인으로 오가던 연결**이 그래프에서는 화살표 하나로 줄어듭니다. 배역(`roles`)처럼 그 연결에만 딸린 값은 **관계의 속성**이 됩니다(표라면 교차 테이블의 열).

## 모델링 원칙: 무엇을 노드·관계·속성으로?
새 도메인을 그래프로 옮길 때 쓰는 간단한 지침입니다.

- **개체(명사)는 노드**: 독립적으로 존재하고 다른 것과 연결되며, 그 자체를 조회하고 싶은 대상(영화·사람·회사).
- **동사(연결)는 관계**: 개체와 개체를 잇는 행위·소속(출연하다·감독하다).
- **수식(딸린 값)은 속성**: 개체나 연결에 딸린 단순 값(개봉연도·배역·점수). 그 값을 기준으로 **따로 조회·연결할 일이 많으면** 속성이 아니라 노드로 승격을 고려합니다.
- **연결에 값이 여러 개 붙으면 그 연결을 노드로**: 리뷰에 별점·시각·근거·출처가 함께 붙는다면 `REVIEWED` 관계 대신 `Review` 노드를 만들어 양쪽에 잇습니다(관계의 실체화).

> 예: '장르'는 영화의 속성으로 둘 수도 있지만, "같은 장르의 영화들"을 자주 묻는다면 **장르 노드**로 만들어 여러 영화가 공유하게 하는 편이 낫습니다.

판단 순서를 한 장으로 정리하면 이렇습니다. 새 정보를 담을 때마다 위에서부터 물어보세요.

<img src="images/model_decision_tree.png" width="760">

## 노드로 올리면 어떤 길이 생기나

바로 앞 장르 이야기를 결과로 확인해 봅시다. Movies 에서는 **영화가 노드**라서, 사람에서 출발해 영화를 거쳐 **다시 사람으로 돌아오는** 길이 있습니다. "같은 영화에 나온 사람" 이 그 길입니다.

In [19]:
# 영화를 거쳐 사람으로 되돌아오는 길입니다(실행만 하세요).
# 사람 -> 영화 <- 사람. 가운데 영화를 거쳐 출발한 쪽과 같은 종류로 돌아온다
# 같은 사람과 여러 편을 함께 찍었으면 이름이 여러 번 나오므로 DISTINCT 로 한 번씩만 받는다
co_actors = run_cypher("MATCH (:Person {name:'Keanu Reeves'})-[:ACTED_IN]->(:Movie)"
                       "<-[:ACTED_IN]-(other:Person) "
                       "RETURN DISTINCT other.name AS name ORDER BY name")
print('Keanu Reeves 와 같은 영화에 나온 사람:', len(co_actors), '명')

Keanu Reeves 와 같은 영화에 나온 사람: 14 명


> 이 길은 **영화가 노드이기 때문에** 생깁니다. 가운데에 거쳐 갈 자리가 있으니까요.

### 좋지 않은 쪽: 값이 속성으로만 있을 때

이번에는 **개봉연도**로 같은 것을 해 봅시다. "The Matrix 와 같은 해에 나온 영화" 입니다. 개봉연도는 `Movie` 노드의 **속성**이라 노드가 아닙니다. 그래서 위와 같은 2홉 왕복을 쓸 수 없습니다.

In [20]:
# 속성으로만 있는 값은 거쳐 갈 자리가 없습니다(실행만 하세요).
# 연도를 거쳐 다른 영화로 가는 길이 그래프에 없으니 2홉 왕복 패턴을 쓸 수 없다
# 그래서 영화를 전부 꺼낸 뒤 파이썬으로 짝지어야 한다(그래프가 아니라 목록을 훑는 것이다)
all_movies = run_cypher("MATCH (m:Movie) RETURN m.title AS title, m.released AS released")
year = next(row['released'] for row in all_movies if row['title'] == 'The Matrix')
same_year = sorted(row['title'] for row in all_movies
                   if row['released'] == year and row['title'] != 'The Matrix')
print(year, '년에 나온 다른 영화:', same_year)

1999 년에 나온 다른 영화: ['Bicentennial Man', 'Snow Falling on Cedars', 'The Green Mile']


> 답은 나왔지만 **방법이 다릅니다.** 앞은 영화 노드 하나만 거쳐 갔는데, 이번에는 **영화를 전부 꺼내** 파이썬으로 짝지었습니다. 연도가 속성이라 그래프 안에 거쳐 갈 자리가 없기 때문입니다. 지금은 38편이라 티가 안 나지만, 3만 편이면 3만 편을 다 꺼내야 합니다.

> "같은 해 영화" 를 자주 묻는다면 연도를 `Year` **노드로 올리면** 됩니다. 그러면 `(영화)-[:RELEASED_IN]->(연도)<-[:RELEASED_IN]-(다른 영화)` 로 앞과 같은 2홉 왕복이 생깁니다. **자주 묻는 질문이 노드로 올릴지를 정한다**는 게 이 뜻입니다.

> **관계가 하나도 붙지 않은 레이블**도 같은 곤란을 겪습니다. 설계표에 `Genre` 를 적어 놓고 영화와 잇는 관계를 두지 않았다고 해 봅시다. 장르 노드를 **그 자체로 꺼내 보는 것은 됩니다.** 안 되는 것은 영화에서 장르로, 장르에서 다른 영화로 **이어서 묻는 것**입니다. 오갈 화살표가 없으니 정작 물으려던 "같은 장르의 영화" 에 답하지 못합니다.

> 그래서 레이블을 하나 두었으면 **거기에 닿는 관계도 하나는** 두어야 합니다. 그래야 그 레이블이 그래프 안에 들어온 값이 됩니다.

## 설계를 글이 아니라 표로 적어 두기
원칙을 읽는 것과 **직접 적어 보는 것**은 다릅니다. 설계는 보통 네 가지를 정하는 일입니다.

1. 어떤 **노드 레이블**을 둘 것인가
2. 어떤 **관계**를 어느 레이블에서 어느 레이블로 놓을 것인가(방향)
3. 어떤 **노드 속성**을 둘 것인가
4. 어떤 **관계 속성**을 둘 것인가

지금 보고 있는 Movies 그래프를 이 네 칸에 그대로 옮겨 보겠습니다. 파이썬 dict 로 적어 두면 눈으로 훑기도 좋고, 나중에 설계를 규칙에 비춰 볼 수도 있습니다.

In [21]:
# 지금 이 그래프의 설계를 그대로 적어 본 것입니다(실행만 하세요).
# 관계는 (주어 레이블, 목적어 레이블) 로 방향까지 적는다. 화살표가 어디서 어디로 가는지가 설계다
movies_model = {
    'nodes': {'Person', 'Movie'},
    'relationships': {
        'ACTED_IN': ('Person', 'Movie'),    # 사람이 영화에 출연한다
        'DIRECTED': ('Person', 'Movie'),
        'PRODUCED': ('Person', 'Movie'),
        'WROTE': ('Person', 'Movie'),
        'REVIEWED': ('Person', 'Movie'),   # 사람이 영화를 평가한다
        'FOLLOWS': ('Person', 'Person'),   # 같은 레이블끼리도 이어진다(2절)
    },
    # 속성은 레이블마다 다르다. 영화에는 name 이 없고 사람에는 title 이 없다(3절에서 확인했다)
    'node_properties': {'Movie': {'title', 'released', 'tagline'},
                        'Person': {'name', 'born'}},
    # 여기가 핵심이다. roles·rating 은 노드가 아니라 '연결' 에 붙는다
    'rel_properties': {'ACTED_IN': {'roles'}, 'REVIEWED': {'rating', 'summary'}},
}
print('레이블:', sorted(movies_model['nodes']), '| 관계:', len(movies_model['relationships']), '종류')

레이블: ['Movie', 'Person'] | 관계: 6 종류


> 앞 절들에서 **하나씩 확인한 것**이 이 표에 그대로 들어 있습니다. 레이블 두 개(1절), 관계 여섯 종류와 그 방향(2절), 노드 속성과 관계 속성의 구분(3절). 설계란 결국 **이 네 칸을 채우는 일**입니다.

> `rel_properties` 를 보세요. `rating` 이 `Movie` 가 아니라 `REVIEWED` 아래에 있습니다. "한 영화의 별점" 이 아니라 "누가 그 영화에 매긴 별점" 이기 때문입니다. **값이 누구마다 달라지면 그 값은 연결에 붙습니다.**

### 🖐️ 함께 따라하기: 도서 도메인 한 조각을 같은 형식으로

이번에는 **책** 이야기입니다. 아래 사실만 담아 위와 같은 형식의 dict `book_sketch` 를 만들고, 레이블 수와 관계 수를 출력하세요.

- 독자(`Reader`)가 책(`Book`)을 **구매한다**(`PURCHASED`).
- 저자(`Author`)가 책을 **집필했다**(`WROTE`).
- 책에는 제목(`title`)이, 독자에게는 이름(`name`)이 있다.
- 독자가 책에 매긴 **별점**(`rating`)도 담는다. 여기서는 **관계를 이 둘로만 두고**, 별점을 **어디에 붙일지만** 위 원칙으로 판단하세요.

**확인 기준**: 레이블 **3개**, 관계 **2종류**가 나옵니다. 별점은 노드가 아니라 **관계 속성** 자리에 있어야 합니다(독자마다 다른 값이니까요).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 위 movies_model 과 같은 네 칸(nodes·relationships·node_properties·rel_properties)을 만든다
# 2) 관계는 (주어 레이블, 목적어 레이블) 로 방향까지 적는다
# 3) 별점 rating 을 어디에 둘지 원칙으로 판단해 넣는다
# 4) 레이블 수와 관계 수를 출력한다

In [22]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 위 movies_model 과 같은 네 칸(nodes·relationships·node_properties·rel_properties)을 만든다
# 2) 관계는 (주어 레이블, 목적어 레이블) 로 방향까지 적는다
# 3) 별점 rating 을 어디에 둘지 원칙으로 판단해 넣는다
# 4) 레이블 수와 관계 수를 출력한다
book_sketch = {
    'nodes': {'Reader', 'Book', 'Author'},
    'relationships': {
        'PURCHASED': ('Reader', 'Book'),   
        'WROTE': ('Author', 'Book'),
    },
    'node_properties': {'Reader': {'name'},
                        'Book': {'title', 'ISBN', 'published_at'}},
    # 여기가 핵심이다. roles·rating 은 노드가 아니라 '연결' 에 붙는다
    'rel_properties': {'PURCHASED': {'rating'}}
}


book_sketch_v2 = {
    'nodes': {'Reader', 'Book', 'Author', 'PUBLISHED_DATE'},
    'relationships': {
        'PURCHASED': ('Reader', 'Book'),   
        'WROTE': ('Author', 'Book'),
        'DATE': ('Book', 'PUBLISHED_DATE'),
        'REIVIEW' : ('Reader', 'Book')
    },
    'node_properties': {'Reader': {'name'},
                        'Book': {'title', 'ISBN'}},
    # 여기가 핵심이다. roles·rating 은 노드가 아니라 '연결' 에 붙는다
    'rel_properties': {'REIVIEW': {'rating', 'summary'}}
}

---
## 이번 강의 정리

| 개념 | 핵심 |
|---|---|
| 노드·레이블 | 점 = 노드, 종류 이름표 = 레이블(`Person`·`Movie`). RDB 의 행·테이블 |
| 관계·방향 | 노드를 잇는 연결. **방향**이 있어 "A→B"와 "B→A"를 구분. 같은 레이블끼리도 이어짐(`FOLLOWS`) |
| 다중 관계 | 한 쌍을 잇는 관계가 여러 개일 수 있고(감독이면서 출연), 관계마다 자기 속성을 가짐 |
| 속성 | 노드에도 **관계에도** 붙는 딸린 값(`roles`·`rating` 은 관계 속성). 없을 수도 있음 |
| RDB 대응 | 관계 = 외래키/조인 테이블, 관계 속성 = 교차 테이블의 추가 열 |
| 모델링 원칙 | 개체=노드·동사=관계·수식=속성. 자주 조회·연결하는 값은 노드로 승격 |
| 설계 적어 두기 | 레이블 · 관계(주어→목적어) · 노드 속성 · 관계 속성 네 칸 |

- **관계에도 속성이 붙는다**는 점이 그래프 모델의 큰 장점입니다(`roles`·`rating`).
- 한 쌍을 여러 관계가 잇는 것도 자연스럽습니다. 표라면 표를 나누고 조인해야 할 일입니다.
- 무엇을 노드·관계·속성으로 둘지는 **자주 던지는 질문**에 달렸습니다.

## ⏭️ 예고: 다음 시간

다음 단원은 **Cypher 기초**입니다. 그래프를 직접 만들고 조회하는 쿼리 언어를 배웁니다(`CREATE`·`MATCH`·`RETURN`). 오늘 이해한 노드·관계·속성 모델이 그 문법의 밑바탕이 됩니다.

수고하셨습니다!